<a href="https://colab.research.google.com/github/sonyxoxo/ipd/blob/main/%D0%BF%D0%B5%D1%80%D0%B2%D0%B8%D1%87%D0%BD%D1%8B%D0%B9_%D0%B0%D1%83%D0%B4%D0%B8%D1%82_%D0%B4%D0%B0%D0%BD%D0%BD%D1%8B%D1%85.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Этап 1. Первичный аудит исходных данных

Исходные измерения открываются **непосредственно из клона GitHub-репозитория**. Файл читается в `pandas` один раз. Ни одна ячейка не загружает файлы из интерфейса Google Colab.

## Параметры и переменные

- `repo_dir` — каталог клонированного репозитория `ipd`, в котором находится общий модуль анализа.
- `source` — полный путь к конкретному исходному TXT/CSV/TSV/ZIP в репозитории.
- `measurements` — исходные измерения в рабочей таблице `pandas`, где даты уже распознаны, а измерения преобразованы в числа.
- `output_dir` — отдельная папка результата именно этого файла; исходник в ней не изменяется.
- `timestamp` — дата и время соответствующего измерения.
- `Global_active_power` — средняя активная мощность за минуту в кВт.
- `valid_Global_active_power` — признак физически допустимого измерения мощности (`True` — корректно).

Если файлов несколько, раскройте одну строку `source = choose_source("data/raw/ИМЯ.csv")` и укажите файл. Для автоматического анализа всех файлов используйте `python run_analysis.py --all`.

Если у нового файла другие заголовки или мощность указана в ваттах, её единицы и названия задаются в `analysis_config.json` в корне репозитория. Это защищает от ошибочного расчёта энергии.

In [1]:
# Одна настройка пути к репозиторию. Исходные файлы не загружаются вручную.
from pathlib import Path
import subprocess
import sys

repo_dir = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "ipd_analysis.py").is_file()), None)
if repo_dir is None:
    repo_dir = Path("/content/ipd")
    if not (repo_dir / "ipd_analysis.py").is_file():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/sonyxoxo/ipd.git", str(repo_dir)], check=True)
sys.path.insert(0, str(repo_dir))

from ipd_analysis import choose_source, load_measurements, regular_minutes, result_folder, save_csv, save_run_metadata

# Если в data/raw несколько файлов, задайте относительный путь явно:
# source = choose_source("data/raw/household_power_consumption.zip")
source = choose_source()
measurements = load_measurements(source)  # Один pd.read_csv на весь ноутбук.
output_dir = result_folder(source)
output_dir.mkdir(parents=True, exist_ok=True)
print("Источник:", source)
print("Строк:", len(measurements))

Источник: /content/ipd/data/raw/household_power_consumption.zip
Строк: 2075259


## Проверка формы и содержания таблицы

- `audit_table` — одна строка контроля: число строк, начало/конец ряда, повторные и неравномерные временные метки, отсутствующая мощность.
- `sample_rows` — первые десять строк, выбранные лишь для просмотра, а не для анализа вместо полного файла.
- `unknown_values` — словарь количества нечисловых значений по каждому исходному измеряемому столбцу. Значения `?` и пустые ячейки становятся `NaN`, а не нулём.

Контрольные показатели исходного UCI (2 075 259 строк, 25 979 отсутствующих мощностей) полезны для **сравнения**, но не блокируют анализ другого файла.

In [2]:
from ipd_analysis import audit

audit_table = audit(measurements)
sample_rows = measurements.head(10)
unknown_values = measurements.attrs["unknown_values"]
display(audit_table)
display(sample_rows)
print("Нечисловые обозначения (кроме стандартных пропусков):", unknown_values)
save_csv(audit_table, output_dir / "01_audit.csv")

,source,rows,start,end,duplicate_timestamps,non_minute_transitions,unknown_active_power,non_numeric_active_power
0,data/raw/household_power_consumption.zip,2075259,2006-12-16 17:24:00,2010-11-26 21:02:00,0,0,25979,0


,timestamp,Global_active_power,valid_Global_active_power,Global_reactive_power,valid_Global_reactive_power,Voltage,valid_Voltage,Global_intensity,valid_Global_intensity,Sub_metering_1,valid_Sub_metering_1,Sub_metering_2,valid_Sub_metering_2,Sub_metering_3,valid_Sub_metering_3
0,2006-12-16 17:24:00,4.216,True,0.418,True,234.84,True,18.4,True,0.0,True,1.0,True,17.0,True
1,2006-12-16 17:25:00,5.360,True,0.436,True,233.63,True,23.0,True,0.0,True,1.0,True,16.0,True
2,2006-12-16 17:26:00,5.374,True,0.498,True,233.29,True,23.0,True,0.0,True,2.0,True,17.0,True
3,2006-12-16 17:27:00,5.388,True,0.502,True,233.74,True,23.0,True,0.0,True,1.0,True,17.0,True
4,2006-12-16 17:28:00,3.666,True,0.528,True,235.68,True,15.8,True,0.0,True,1.0,True,17.0,True
5,2006-12-16 17:29:00,3.520,True,0.522,True,235.02,True,15.0,True,0.0,True,2.0,True,17.0,True
6,2006-12-16 17:30:00,3.702,True,0.520,True,235.09,True,15.8,True,0.0,True,1.0,True,17.0,True
7,2006-12-16 17:31:00,3.700,True,0.520,True,235.22,True,15.8,True,0.0,True,1.0,True,17.0,True
8,2006-12-16 17:32:00,3.668,True,0.510,True,233.99,True,15.8,True,0.0,True,1.0,True,17.0,True
9,2006-12-16 17:33:00,3.662,True,0.510,True,233.86,True,15.8,True,0.0,True,2.0,True,16.0,True


Нечисловые обозначения (кроме стандартных пропусков): {'Global_active_power': 0, 'Global_reactive_power': 0, 'Voltage': 0, 'Global_intensity': 0, 'Sub_metering_1': 0, 'Sub_metering_2': 0, 'Sub_metering_3': 0}


PosixPath('/content/ipd/результаты/household_power_consumption-b916e64b/01_audit.csv')

## Проверка временной сетки

- `minutes` — таблица с **каждой ожидаемой минутой** между началом и концом записи.
- `missing_timestamp_rows` — сколько минут целиком отсутствует в исходном файле. Это **не то же самое**, что строка с `NaN` в мощности.
- `missing_measurements` — сколько минут не имеют пригодной активной мощности после учёта обоих видов пропусков.

Повторяющиеся временные метки сначала нужно исследовать: программа сообщит об ошибке, а не удалит повторы без вашего решения.

In [3]:
minutes = regular_minutes(measurements)
missing_timestamp_rows = len(minutes) - len(measurements)
missing_measurements = int((~minutes["valid_Global_active_power"]).sum())
print("Пропущенных временных меток:", missing_timestamp_rows)
print("Минут без корректного измерения:", missing_measurements)

Пропущенных временных меток: 0
Минут без корректного измерения: 25979


## Воспроизводимость результата

Последняя ячейка сохраняет `metadata.json` с SHA-256 исходного файла, хешами исполняемого кода, версиями библиотек и датой запуска. Поэтому любой результат можно связать с конкретным источником и версией обработки.

In [4]:
save_run_metadata(source, output_dir, {"notebook": 'первичный аудит данных.ipynb'})

PosixPath('/content/ipd/результаты/household_power_consumption-b916e64b/metadata.json')